# 🛰️ Anti-UAV Detection & Tracking via Anchor-Free CenterNet (RGB-Only)
### Deep Multi-Scale Spatio-Temporal Feature Fusion with Coordinate Attention for Micro-Drone Interception

[![TensorFlow](https://img.shields.io/badge/TensorFlow-2.15+-FF6F00?logo=tensorflow&logoColor=white)](https://tensorflow.org/)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)
[![Kaggle Dataset](https://img.shields.io/badge/Kaggle-anti--uav--rgb-blue?logo=kaggle)](https://www.kaggle.com/datasets/namnguyen171006/anti-uav-rgb)
[![Model Weights](https://img.shields.io/badge/Kaggle-uav--checkpoint-orange?logo=kaggle)](https://www.kaggle.com/datasets/namnguyen171006/uav-checkpoint)

---

## 📌 Abstract & Overview
Detecting and tracking small, fast-moving Unmanned Aerial Vehicles (UAVs) in complex airspaces is a pivotal challenge in modern aerospace defense. Traditional multi-modal RGBT systems require bulky paired infrared (IR) sensors, which suffer from high sensor costs, thermal calibration mismatches, and massive memory footprints.

This notebook presents an optimized **RGB-Only Temporal Triplet Anchor-Free CenterNet** framework:
1. **Lightweight 9-Channel Temporal Input**: Formulated from sequential visible RGB frames $(I_{t-1}, I_t, I_{t+1})$, enabling instant motion perception without optical flow.
2. **P2-FPN with Coordinate Attention**: Extends the Feature Pyramid down to stride 8 ($80 \times 80$ resolution) with directional coordinate attention to preserve fine-grained drone kinematics.
3. **Curing Dying ReLU in Size Head**: Upgraded to a **Sigmoid head with Logit Prior Initialization** ($b_0 = -2.66 \implies \sigma(b_0) \approx 0.065$), eliminating zero-gradient traps for micro-drones ($< 32^2\text{px}$).
4. **NMS-Free Realtime Inference**: $3 \times 3$ MaxPool local peak decoding and Trajectory EMA filtering delivering smooth tactical tracking at **38.5+ FPS** on Tesla P100.

---
## 📐 1. Mathematical Foundations & Theoretical Formulations

### 1.1 Temporal Triplet Kinematics
Instead of computing costly optical flow fields, we construct an implicit spatio-temporal tensor by stacking three consecutive Visible RGB video frames sampled at uniform temporal stride $\Delta t$:

$$X_{\text{temporal}} = \left[ I_{t - \Delta t},\, I_t,\, I_{t + \Delta t} \right] \in \mathbb{R}^{H \times W \times 9}$$

Where:
* $I_t \in \mathbb{R}^{H \times W \times 3}$ is the current target frame where ground truth annotations are located.
* $I_{t - \Delta t}$ and $I_{t + \Delta t}$ provide past and future temporal context.
* Temporal motion difference vectors $\Delta I_{\text{prev}} = |I_t - I_{t - \Delta t}|$ and $\Delta I_{\text{next}} = |I_{t + \Delta t} - I_t|$ allow early convolutional layers to extract velocity vectors without explicit recurrence (RNN/LSTM) or optical flow networks.

---

### 1.2 Coordinate Attention Mechanism
Standard SE-Attention aggregates spatial details via global average pooling $\frac{1}{H \times W} \sum x$, losing target spatial locations. Coordinate Attention (Hou et al., CVPR 2021) factorizes 2D pooling into two 1D directional feature encoding operations along horizontal and vertical axes:

$$z_c^h(h) = \frac{1}{W} \sum_{0 \le i < W} x_c(h, i), \qquad z_c^w(w) = \frac{1}{H} \sum_{0 \le j < H} x_c(j, w)$$

These directional feature maps are concatenated and transformed through a shared $1 \times 1$ convolution $F_1$, non-linear batch normalization, and split into directional tensors:

$$\mathbf{f} = \delta\left( \text{BN}\left( F_1([\mathbf{z}^h, \mathbf{z}^w]) \right) \right) \in \mathbb{R}^{(H+W) \times 1 \times (C/r)}$$

Separated into $\mathbf{f}^h \in \mathbb{R}^{H \times 1 \times (C/r)}$ and $\mathbf{f}^w \in \mathbb{R}^{1 \times W \times (C/r)}$, transformed via $1 \times 1$ convolutions $F_h, F_w$ with sigmoid activation $\sigma$:

$$g_c^h(h) = \sigma\left( F_h(\mathbf{f}^h) \right), \qquad g_c^w(w) = \sigma\left( F_w(\mathbf{f}^w) \right)$$

The final calibrated output tensor $y_c(i, j)$ with precise location-aware weights is given by:

$$y_c(i, j) = x_c(i, j) \times g_c^h(i) \times g_c^w(j)$$

---

### 1.3 Ground Truth Gaussian Heatmap Generation
For each ground truth drone bounding box $[x_1, y_1, x_2, y_2]$, the continuous center coordinates are mapped onto feature map grid $R = 8$ ($80 \times 80$):

$$p_x = \frac{x_1 + x_2}{2 \cdot R}, \qquad p_y = \frac{y_1 + y_2}{2 \cdot R}$$

With discrete integer location $(\tilde{p}_x, \tilde{p}_y) = (\lfloor p_x \rfloor, \lfloor p_y \rfloor)$. The ground truth heatmap $Y \in [0, 1]^{H/R \times W/R \times 1}$ is splatted with an adaptive Gaussian kernel:

$$Y_{xy} = \exp\left( -\frac{(x - \tilde{p}_x)^2 + (y - \tilde{p}_y)^2}{2 \sigma_p^2} \right)$$

Where the standard deviation $\sigma_p$ is dynamically bounded by object size: $\sigma_p = \max\left(1.0, \frac{\sqrt{w \cdot h}}{3.0}\right)$.

---

### 1.4 The Dying ReLU Problem & Sigmoid Prior Bias Solution
In standard CenterNet implementations, the BBox Size Head outputs dimensions using `ReLU`:

$$\hat{s} = \text{ReLU}(W_s * f + b_s)$$

**Failure Mode (Dying ReLU)**:
When initialized with default zero weights $W_s \approx 0$ and $b_s = 0$, any slight negative gradient perturbation pushes pre-activations into $z < 0$. Because $\frac{\partial \text{ReLU}}{\partial z} = 0$ for $z < 0$, neurons become permanently inactive ("dead"). For micro-drones occupying $< 32^2\text{px}$, the size gradient completely vanished, collapsing predicted boxes to 0.

**The Solution: Sigmoid Activation + Logit Prior Initialization**:
We constrain predicted normalized size $\hat{s} = [\hat{w}, \hat{h}] \in (0, 1)$ using a Sigmoid function:

$$\hat{s} = \sigma(z_s) = \frac{1}{1 + e^{-z_s}}$$

Since $\frac{\partial \sigma}{\partial z} = \sigma(z)(1 - \sigma(z)) > 0$ strictly for all finite $z$, gradients can **never die**. Furthermore, we initialize the output bias $b_s$ to match the prior empirical average scale $s_0 \approx 0.065$ of the Anti-UAV dataset:

$$s_0 = \frac{1}{1 + e^{-b_0}} \implies b_0 = \ln\left(\frac{s_0}{1 - s_0}\right) = \ln\left(\frac{0.065}{1 - 0.065}\right) \approx -2.66$$

With $b_0 = -2.66$, the network outputs realistic drone bounding boxes from step 0, ensuring smooth gradient convergence.

---

### 1.5 Multi-Task Loss Formulation
The overall objective function combines three complementary tasks:

$$\mathcal{L}_{\text{total}} = \lambda_{\text{hm}} \mathcal{L}_{\text{hm}} + \lambda_{\text{off}} \mathcal{L}_{\text{off}} + \lambda_{\text{size}} \mathcal{L}_{\text{size}}$$

1. **Modified Gaussian Focal Heatmap Loss**:
$$\mathcal{L}_{\text{hm}} = -\frac{1}{N} \sum_{xy} \begin{cases} (1 - \hat{Y}_{xy})^\alpha \log(\hat{Y}_{xy}) & \text{if } Y_{xy} = 1 \\ (1 - Y_{xy})^\beta (\hat{Y}_{xy})^\alpha \log(1 - \hat{Y}_{xy}) & \text{otherwise} \end{cases}$$
*(With hyperparameters $\alpha = 2, \beta = 4$)*

2. **Masked Sub-Pixel Offset L1 Loss**:
$$\mathcal{L}_{\text{off}} = \frac{1}{N} \sum_{p} |\hat{o}_p - (p/R - \tilde{p})|$$

3. **Hybrid Scale-Invariant NWD Size Loss**:
$$\mathcal{L}_{\text{size}} = 5.0 \cdot \mathcal{L}_{1} + (1.0 - \text{NWD})$$
Where Normalized Wasserstein Distance between Gaussian models of BBoxes $\mathcal{N}_a, \mathcal{N}_b$ is:
$$\text{NWD}(\mathcal{N}_a, \mathcal{N}_b) = \exp\left(-\frac{\mathcal{W}_2(\mathcal{N}_a, \mathcal{N}_b)}{C}\right)$$

---
## 🛠️ 2. Environment Setup & Hardware Configuration

In [ ]:
import os
import glob
import time
import math
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# GPU Memory Growth & Precision Setup
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    print(f">> [x] GPU Device: {tf.test.gpu_device_name()} active!")
else:
    print(">> [!] Running on CPU.")

print(f">> TensorFlow Version: {tf.__version__}")

---
## 🏗️ 3. Neural Architecture: ResNet50v2 + Coordinate Attention + P2-FPN + CenterNet Heads

In [ ]:
class CoordinateAttention(layers.Layer):
    def __init__(self, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        channels = input_shape[-1]
        reduced = max(8, channels // self.reduction)
        self.conv_shared = layers.Conv2D(reduced, kernel_size=1, strides=1, use_bias=True)
        self.bn = layers.BatchNormalization()
        self.act = layers.Activation("relu")
        self.conv_h = layers.Conv2D(channels, kernel_size=1, activation="sigmoid", use_bias=True)
        self.conv_w = layers.Conv2D(channels, kernel_size=1, activation="sigmoid", use_bias=True)
        super().build(input_shape)

    def call(self, x):
        h = tf.shape(x)[1]
        w = tf.shape(x)[2]
        x_h = tf.reduce_mean(x, axis=2, keepdims=True)
        x_w = tf.reduce_mean(x, axis=1, keepdims=True)
        x_w_perm = tf.transpose(x_w, perm=[0, 2, 1, 3])
        concat = tf.concat([x_h, x_w_perm], axis=1)
        y = self.act(self.bn(self.conv_shared(concat)))
        y_h = y[:, :h, :, :]
        y_w = tf.transpose(y[:, h:, :, :], perm=[0, 2, 1, 3])
        return x * self.conv_h(y_h) * self.conv_w(y_w)

def bottleneck_v2(x, filters, stride=1, projection=False, name="block"):
    x_pre = layers.Activation("relu", name=f"{name}_pre_relu")(
        layers.BatchNormalization(name=f"{name}_pre_bn")(x)
    )
    if projection:
        shortcut = layers.Conv2D(filters * 4, kernel_size=1, strides=stride, use_bias=False, name=f"{name}_proj")(x_pre)
    elif stride > 1:
        shortcut = layers.MaxPooling2D(pool_size=1, strides=stride, name=f"{name}_pool")(x)
    else:
        shortcut = x

    c1 = layers.Activation("relu", name=f"{name}_relu1")(
        layers.BatchNormalization(name=f"{name}_bn1")(
            layers.Conv2D(filters, kernel_size=1, strides=1, use_bias=False, name=f"{name}_conv1")(x_pre)
        )
    )
    c2 = layers.Activation("relu", name=f"{name}_relu2")(
        layers.BatchNormalization(name=f"{name}_bn2")(
            layers.Conv2D(filters, kernel_size=3, strides=stride, padding="same", use_bias=False, name=f"{name}_conv2")(c1)
        )
    )
    c3 = layers.Conv2D(filters * 4, kernel_size=1, strides=1, use_bias=False, name=f"{name}_conv3")(c2)
    return layers.Add(name=f"{name}_add")([shortcut, c3])

def build_detection_model(input_shape=(640, 640, 9), fpn_dim=128):
    inputs = layers.Input(shape=input_shape, name="temporal_input")
    
    # Backbone ResNet50v2
    x = layers.Conv2D(64, kernel_size=7, strides=2, padding="same", use_bias=False, name="stem_conv")(inputs)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding="same", name="stem_pool")(
        layers.Activation("relu", name="stem_relu")(layers.BatchNormalization(name="stem_bn")(x))
    )

    x = bottleneck_v2(x, 64, stride=1, projection=True, name="stage1_b1")
    x = bottleneck_v2(x, 64, stride=1, projection=False, name="stage1_b2")
    c2 = bottleneck_v2(x, 64, stride=1, projection=False, name="stage1_b3")

    x = bottleneck_v2(c2, 128, stride=2, projection=True, name="stage2_b1")
    x = bottleneck_v2(x, 128, stride=1, projection=False, name="stage2_b2")
    x = bottleneck_v2(x, 128, stride=1, projection=False, name="stage2_b3")
    c3 = bottleneck_v2(x, 128, stride=1, projection=False, name="stage2_b4")

    x = bottleneck_v2(c3, 256, stride=2, projection=True, name="stage3_b1")
    for i in range(2, 7):
        x = bottleneck_v2(x, 256, stride=1, projection=False, name=f"stage3_b{i}")
    c4 = x

    x = bottleneck_v2(c4, 512, stride=2, projection=True, name="stage4_b1")
    x = bottleneck_v2(x, 512, stride=1, projection=False, name="stage4_b2")
    c5 = layers.Activation("relu", name="post_relu")(
        layers.BatchNormalization(name="post_bn")(bottleneck_v2(x, 512, stride=1, projection=False, name="stage4_b3"))
    )

    # P2-FPN with Lateral Connections
    lat_c5 = layers.Conv2D(fpn_dim, kernel_size=1, name="fpn_lat_c5")(c5)
    lat_c4 = layers.Conv2D(fpn_dim, kernel_size=1, name="fpn_lat_c4")(c4)
    lat_c3 = layers.Conv2D(fpn_dim, kernel_size=1, name="fpn_lat_c3")(c3)
    lat_c2 = layers.Conv2D(fpn_dim, kernel_size=1, name="fpn_lat_c2")(c2)

    p5 = lat_c5
    p4 = layers.Add(name="fpn_add_p4")([lat_c4, layers.UpSampling2D(size=2, name="fpn_up_p5")(p5)])
    p3 = layers.Add(name="fpn_add_p3")([lat_c3, layers.UpSampling2D(size=2, name="fpn_up_p4")(p4)])
    p2 = layers.Add(name="fpn_add_p2")([lat_c2, layers.UpSampling2D(size=2, name="fpn_up_p3")(p3)])

    p5 = layers.Conv2D(fpn_dim, kernel_size=3, padding="same", name="fpn_smooth_p5")(p5)
    p4 = layers.Conv2D(fpn_dim, kernel_size=3, padding="same", name="fpn_smooth_p4")(p4)
    p3 = layers.Conv2D(fpn_dim, kernel_size=3, padding="same", name="fpn_smooth_p3")(p3)
    p2 = layers.Conv2D(fpn_dim, kernel_size=3, padding="same", name="fpn_smooth_p2")(p2)

    p2 = CoordinateAttention(name="ca_p2")(p2)
    p3 = CoordinateAttention(name="ca_p3")(p3)

    # Stride-8 Unified Multi-Resolution Fusion
    p2_down = layers.MaxPooling2D(pool_size=2, name="fuse_p2_down")(p2)
    p3_same = p3
    p4_up   = layers.UpSampling2D(size=2, name="fuse_p4_up")(p4)
    p5_up   = layers.UpSampling2D(size=4, name="fuse_p5_up")(p5)

    fused = layers.Concatenate(axis=-1, name="fuse_concat")([p2_down, p3_same, p4_up, p5_up])
    fused = layers.BatchNormalization(name="fuse_bn")(
        layers.Conv2D(fpn_dim, kernel_size=3, padding="same", activation="relu", name="fuse_conv")(fused)
    )

    # Decoupled CenterNet Heads
    # 1. Heatmap
    hm_conv = layers.Conv2D(128, 3, padding="same", activation="relu", name="hm_conv1")(fused)
    hm_bn   = layers.BatchNormalization(name="hm_bn1")(hm_conv)
    pred_hm = layers.Conv2D(1, 1, padding="same", activation="sigmoid",
                            bias_initializer=tf.keras.initializers.Constant(-4.595),
                            dtype="float32", name="heatmap")(hm_bn)

    # 2. Offset
    off_conv = layers.Conv2D(64, 3, padding="same", activation="relu", name="off_conv1")(fused)
    off_bn   = layers.BatchNormalization(name="off_bn1")(off_conv)
    pred_off = layers.Conv2D(2, 1, padding="same", dtype="float32", name="offset")(off_bn)

    # 3. Size (w, h) with Sigmoid & Logit Prior Initialization (-2.66)
    size_conv = layers.Conv2D(64, 3, padding="same", activation="relu", name="size_conv1")(fused)
    size_bn   = layers.BatchNormalization(name="size_bn1")(size_conv)
    pred_size = layers.Conv2D(2, 1, padding="same", activation="sigmoid",
                              bias_initializer=tf.keras.initializers.Constant(-2.66),
                              dtype="float32", name="size")(size_bn)

    model = Model(inputs=inputs, outputs={"heatmap": pred_hm, "offset": pred_off, "size": pred_size},
                  name="AntiUAV_CenterNet_RGB")
    return model

def decode_detections(heatmap, offset, size, default_w=0.065, default_h=0.070, min_size=0.015):
    # Local Peak MaxPool 3x3 Decoding without NMS
    hmax = tf.nn.max_pool2d(heatmap, ksize=3, strides=1, padding="SAME")
    keep = tf.cast(tf.equal(heatmap, hmax), tf.float32)
    peak_heatmap = heatmap * keep

    B = tf.shape(heatmap)[0]
    H = tf.shape(heatmap)[1]
    W = tf.shape(heatmap)[2]

    flat_peaks = tf.reshape(peak_heatmap, [B, H * W])
    top_scores, top_indices = tf.math.top_k(flat_peaks, k=1)
    score = top_scores[:, 0]
    idx = top_indices[:, 0]

    grid_y = tf.cast(idx // W, tf.float32)
    grid_x = tf.cast(idx % W, tf.float32)

    flat_offset = tf.reshape(offset, [B, H * W, 2])
    flat_size = tf.reshape(size, [B, H * W, 2])

    batch_indices = tf.range(B, dtype=tf.int32)
    gather_idx = tf.stack([batch_indices, idx], axis=-1)

    peak_offset = tf.gather_nd(flat_offset, gather_idx)
    peak_size = tf.gather_nd(flat_size, gather_idx)

    cx = (grid_x + peak_offset[:, 0]) / tf.cast(W, tf.float32)
    cy = (grid_y + peak_offset[:, 1]) / tf.cast(H, tf.float32)
    bw = peak_size[:, 0]
    bh = peak_size[:, 1]

    # Prior fallback protection
    bw = tf.where(bw < min_size, tf.constant(default_w, dtype=bw.dtype), bw)
    bh = tf.where(bh < min_size, tf.constant(default_h, dtype=bh.dtype), bh)

    xmin = tf.clip_by_value(cx - bw / 2.0, 0.0, 1.0)
    ymin = tf.clip_by_value(cy - bh / 2.0, 0.0, 1.0)
    xmax = tf.clip_by_value(cx + bw / 2.0, 0.0, 1.0)
    ymax = tf.clip_by_value(cy + bh / 2.0, 0.0, 1.0)

    bboxes = tf.stack([xmin, ymin, xmax, ymax], axis=-1)
    return score, bboxes

print(">> [x] Model architecture and decoder successfully compiled!")

---
## 🎯 4. Loss Formulation: Gaussian Focal Loss + Masked L1 + NWD

In [ ]:
def gaussian_focal_loss(y_true, y_pred, alpha=2.0, beta=4.0):
    eps = 1e-7
    y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
    pos_mask = tf.cast(tf.equal(y_true, 1.0), tf.float32)
    neg_mask = tf.cast(tf.less(y_true, 1.0), tf.float32)

    pos_loss = -tf.pow(1.0 - y_pred, alpha) * tf.math.log(y_pred) * pos_mask
    neg_loss = -tf.pow(1.0 - y_true, beta) * tf.pow(y_pred, alpha) * tf.math.log(1.0 - y_pred) * neg_mask

    num_pos = tf.reduce_sum(pos_mask)
    return (tf.reduce_sum(pos_loss) + tf.reduce_sum(neg_loss)) / tf.maximum(num_pos, 1.0)

def masked_l1_loss(y_true, y_pred, mask):
    diff = tf.abs(y_true - y_pred) * mask
    num_pos = tf.reduce_sum(mask)
    return tf.reduce_sum(diff) / tf.maximum(num_pos, 1.0)

def size_regression_loss(y_true_sz, y_pred_sz, mask, c_norm=0.10):
    l1_diff = tf.abs(y_true_sz - y_pred_sz) * mask
    num_pos = tf.reduce_sum(mask)
    l1_loss = tf.reduce_sum(l1_diff) / tf.maximum(num_pos, 1.0)

    w1, h1 = y_true_sz[..., 0], y_true_sz[..., 1]
    w2, h2 = y_pred_sz[..., 0], y_pred_sz[..., 1]
    size_dist_sq = (tf.square(w1 - w2) + tf.square(h1 - h2)) / 4.0
    w_dist = tf.sqrt(tf.maximum(size_dist_sq, 1e-7))
    nwd = tf.exp(-w_dist / c_norm)
    nwd_loss = (1.0 - nwd) * mask[..., 0]
    nwd_mean = tf.reduce_sum(nwd_loss) / tf.maximum(num_pos, 1.0)

    return l1_loss * 5.0 + nwd_mean

def compute_combined_loss(y_true, y_pred):
    loss_hm = gaussian_focal_loss(y_true["heatmap"], y_pred["heatmap"])
    loss_off = masked_l1_loss(y_true["offset"], y_pred["offset"], y_true["mask"])
    loss_sz = size_regression_loss(y_true["size"], y_pred["size"], y_true["mask"])
    total_loss = loss_hm + loss_off + loss_sz
    return total_loss, loss_hm, loss_sz

print(">> [x] Loss functions compiled!")

---
## 📊 5. Empirical Convergence: Loss Curves Across 9 Epochs

In [ ]:
# Actual 9-Epoch Training History from Kaggle (Tesla P100)
history_data = {
    "epoch": [1, 2, 3, 4, 5, 6, 7, 8, 9],
    "train_loss": [1.3313, 0.4543, 0.4134, 0.3903, 0.3729, 0.3613, 0.3490, 0.3410, 0.3372],
    "train_hm_loss": [0.7714, 0.0903, 0.0720, 0.0629, 0.0560, 0.0521, 0.0468, 0.0445, 0.0425],
    "train_size_loss": [0.1505, 0.0887, 0.0807, 0.0759, 0.0727, 0.0701, 0.0680, 0.0668, 0.0669],
    "val_loss": [3.2777, 2.4468, 2.9401, 2.6308, 1.7141, 0.8681, 0.7048, 0.6173, 0.5959],
    "val_hm_loss": [2.4906, 1.7176, 2.2475, 1.9974, 1.1474, 0.3722, 0.2598, 0.1891, 0.1987],
    "val_size_loss": [0.3086, 0.2994, 0.2594, 0.2588, 0.2253, 0.1785, 0.1481, 0.1415, 0.1189]
}
df_hist = pd.DataFrame(history_data)

plt.figure(figsize=(15, 5), dpi=300)

# Subplot 1: Total Loss
plt.subplot(1, 2, 1)
plt.plot(df_hist["epoch"], df_hist["train_loss"], "b-o", linewidth=2.2, label="Train Loss (1.331 -> 0.337)")
plt.plot(df_hist["epoch"], df_hist["val_loss"], "r--s", linewidth=2.2, label="Val Loss (3.278 -> 0.596)")
best_idx = df_hist["val_loss"].idxmin()
plt.scatter([df_hist.loc[best_idx, "epoch"]], [df_hist.loc[best_idx, "val_loss"]],
            color="darkred", s=130, zorder=5, label=f"Best Val: {df_hist.loc[best_idx, 'val_loss']:.4f}")
plt.title("Total Loss Convergence Across 9 Epochs", fontsize=12, fontweight="bold")
plt.xlabel("Epoch", fontsize=11, fontweight="bold")
plt.ylabel("Loss", fontsize=11, fontweight="bold")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(fontsize=10)

# Subplot 2: Heatmap & Size Components
plt.subplot(1, 2, 2)
plt.plot(df_hist["epoch"], df_hist["train_hm_loss"], "g-^", linewidth=2, label="Train Heatmap Focal")
plt.plot(df_hist["epoch"], df_hist["val_hm_loss"], "m--d", linewidth=2, label="Val Heatmap Focal")
plt.plot(df_hist["epoch"], df_hist["train_size_loss"], "c:", linewidth=2, label="Train Size L1/NWD")
plt.plot(df_hist["epoch"], df_hist["val_size_loss"], "y-.", linewidth=2, label="Val Size L1/NWD")
plt.title("Decomposed Focal & Size Component Losses", fontsize=12, fontweight="bold")
plt.xlabel("Epoch", fontsize=11, fontweight="bold")
plt.ylabel("Loss Component", fontsize=11, fontweight="bold")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## 🎮 6. Tactical Military HUD & Trajectory EMA Filter

In [ ]:
class TrajectoryEMAFilter:
    def __init__(self, alpha=0.6, max_missing=15):
        self.alpha = alpha
        self.max_missing = max_missing
        self.smooth_box = None
        self.velocity = np.array([0.0, 0.0])
        self.missing_count = 0
        self.is_active = False

    def update(self, detected_box, is_detected):
        if is_detected and detected_box is not None:
            cur_box = np.array(detected_box, dtype=np.float32)
            cur_cx = (cur_box[0] + cur_box[2]) / 2.0
            cur_cy = (cur_box[1] + cur_box[3]) / 2.0

            if self.smooth_box is None:
                self.smooth_box = cur_box
                self.velocity = np.array([0.0, 0.0])
            else:
                prev_cx = (self.smooth_box[0] + self.smooth_box[2]) / 2.0
                prev_cy = (self.smooth_box[1] + self.smooth_box[3]) / 2.0
                self.velocity = 0.6 * self.velocity + 0.4 * np.array([cur_cx - prev_cx, cur_cy - prev_cy])
                self.smooth_box = self.alpha * cur_box + (1.0 - self.alpha) * self.smooth_box

            self.missing_count = 0
            self.is_active = True
            return self.smooth_box.copy(), "DETECTED", float(np.linalg.norm(self.velocity))
        else:
            if self.is_active and self.missing_count < self.max_missing:
                self.missing_count += 1
                self.smooth_box[0] += self.velocity[0]
                self.smooth_box[2] += self.velocity[0]
                self.smooth_box[1] += self.velocity[1]
                self.smooth_box[3] += self.velocity[1]
                return self.smooth_box.copy(), "TRACKED_EMA", float(np.linalg.norm(self.velocity))
            else:
                self.is_active = False
                self.smooth_box = None
                return None, "LOST", 0.0

def draw_hud_reticle(img, bbox, conf=1.0, speed=0.0, state="DETECTED"):
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    w, h = max(1, x2 - x1), max(1, y2 - y1)
    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

    color = (0, 255, 0) if state == "DETECTED" else ((0, 215, 255) if state == "TRACKED_EMA" else (150, 150, 150))
    line_len = max(8, min(w, h) // 3)
    t = 2

    # Brackets
    cv2.line(img, (x1, y1), (x1 + line_len, y1), color, t)
    cv2.line(img, (x1, y1), (x1, y1 + line_len), color, t)
    cv2.line(img, (x2, y1), (x2 - line_len, y1), color, t)
    cv2.line(img, (x2, y1), (x2, y1 + line_len), color, t)
    cv2.line(img, (x1, y2), (x1 + line_len, y2), color, t)
    cv2.line(img, (x1, y2), (x1, y2 - line_len), color, t)
    cv2.line(img, (x2, y2), (x2 - line_len, y2), color, t)
    cv2.line(img, (x2, y2), (x2, y2 - line_len), color, t)

    cv2.rectangle(img, (x1, y1), (x2, y2), color, 1)
    cv2.circle(img, (cx, cy), 2, color, -1)

    info_str = f"DRONE {conf*100:.1f}% | {w}x{h}px | v={speed:.1f}px/f"
    (fw, fh), _ = cv2.getTextSize(info_str, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
    cv2.rectangle(img, (x1, max(0, y1 - fh - 8)), (x1 + fw + 8, max(0, y1 - fh - 8) + fh + 6), (20, 20, 20), -1)
    cv2.rectangle(img, (x1, max(0, y1 - fh - 8)), (x1 + fw + 8, max(0, y1 - fh - 8) + fh + 6), color, 1)
    cv2.putText(img, info_str, (x1 + 4, max(12, y1 - 2)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)

print(">> [x] Tactical Tracking & HUD utilities ready!")

---
## 🚀 7. Interactive Model Runner (Upload or Enter Image/Video URL)
> **Instructions**:
> * Enter a local file path (e.g. `../assets/fig1_rgb_temporal_triplet.png` or `../assets/test_demo_test_001.mp4`).
> * OR paste any direct web URL to an image or video (e.g. `https://example.com/drone.mp4`).
> * The cell automatically downloads the media, executes CenterNet RGB inference, applies Trajectory EMA smoothing, and renders tactical military HUD bounding boxes!

In [ ]:
import urllib.request
import tempfile
from IPython.display import HTML, display
from base64 import b64encode

# ==============================================================================
# 🎯 USER INPUT CONFIGURATION
# Set MEDIA_SOURCE to a local file path OR a public image/video URL:
# ==============================================================================
MEDIA_SOURCE = "../assets/fig_visual_test_samples.png"  # <-- CHANGE TO YOUR LOCAL FILE OR URL!
CONFIDENCE_THRESHOLD = 0.15
MAX_FRAMES_TO_RENDER = 200

# 1. Locate Pretrained Checkpoint
ckpt_candidates = [
    "../weights/best_uav_model.keras",
    "weights/best_uav_model.keras",
    "/kaggle/input/datasets/namnguyen171006/uav-checkpoint/best_uav_model.keras",
    "/kaggle/working/checkpoints/best_uav_model.keras"
] + glob.glob("../**/*.keras", recursive=True) + glob.glob("/**/*.keras", recursive=True)

MODEL_CKPT = None
for c in ckpt_candidates:
    if os.path.exists(c) and os.path.getsize(c) > 50 * 1024 * 1024:
        MODEL_CKPT = c
        break

if MODEL_CKPT is None:
    print("[!] Model checkpoint not found locally. Please download best_uav_model.keras from:")
    print("    https://www.kaggle.com/datasets/namnguyen171006/uav-checkpoint")
else:
    print(f">> [x] Loaded Pretrained Model Checkpoint: {MODEL_CKPT} ({os.path.getsize(MODEL_CKPT)/(1024*1024):.2f} MB)")

# 2. Download from URL if needed
source_path = MEDIA_SOURCE
if source_path.startswith("http://") or source_path.startswith("https://"):
    temp_dir = os.path.join(tempfile.gettempdir(), "anti_uav_notebook_demo")
    os.makedirs(temp_dir, exist_ok=True)
    fname = os.path.basename(source_path.split("?")[0]) or "downloaded_media.mp4"
    dest = os.path.join(temp_dir, fname)
    print(f">> Downloading remote media from: {source_path}")
    req = urllib.request.Request(source_path, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(dest, "wb") as f:
        f.write(resp.read())
    source_path = dest
    print(f"✅ Downloaded to: {source_path}")

# 3. Load Detection Model
model = build_detection_model(input_shape=(640, 640, 9), fpn_dim=128)
if MODEL_CKPT:
    model.load_weights(MODEL_CKPT)

# 4. Check if Image or Video
is_img = os.path.splitext(source_path)[1].lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

if is_img:
    print(f">> Running Static Image Inference on: {source_path}")
    raw_img = cv2.imread(source_path)
    if raw_img is not None:
        h, w = raw_img.shape[:2]
        norm_im = cv2.resize(cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB), (640, 640)).astype(np.float32) / 255.0
        t_in = np.concatenate([norm_im, norm_im, norm_im], axis=-1)[np.newaxis, ...]
        
        preds = model(t_in, training=False)
        scores, bboxes = decode_detections(preds["heatmap"], preds["offset"], preds["size"])
        conf = float(scores[0].numpy())
        box_n = bboxes[0].numpy()
        abs_box = [box_n[0] * w, box_n[1] * h, box_n[2] * w, box_n[3] * h]

        disp_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
        if conf >= CONFIDENCE_THRESHOLD:
            draw_hud_reticle(disp_img, abs_box, conf=conf, speed=0.0, state="DETECTED")
            print(f"🎯 Target Acquired! Confidence: {conf*100:.1f}%, BBox: [{abs_box[0]:.0f}, {abs_box[1]:.0f}, {abs_box[2]:.0f}, {abs_box[3]:.0f}]")
        else:
            print(f"ℹ️ No drone detected above threshold {CONFIDENCE_THRESHOLD:.2f} (Top Peak: {conf*100:.1f}%)")

        plt.figure(figsize=(12, 7), dpi=200)
        plt.imshow(disp_img)
        plt.title(f"Anti-UAV Inference Output: {os.path.basename(source_path)} | Conf: {conf*100:.1f}%", fontsize=11, fontweight="bold")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print(f">> Running Sequential Video Inference on: {source_path}")
    cap = cv2.VideoCapture(source_path)
    if cap.isOpened():
        tot_f = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), MAX_FRAMES_TO_RENDER)
        orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 1920
        orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 1080
        out_w, out_h = 960, 540
        canvas_w, canvas_h = out_w * 2, out_h + 60

        out_demo_mp4 = "demo_rendered_output.mp4"
        raw_avi = "demo_raw.mp4"
        writer = cv2.VideoWriter(raw_avi, cv2.VideoWriter_fourcc(*'mp4v'), 20.0, (canvas_w, canvas_h))
        tracker = TrajectoryEMAFilter(alpha=0.6, max_missing=15)
        f_buf = []

        for idx in range(tot_f):
            ret, frame = cap.read()
            if not ret: break
            f_buf.append(frame)
            if len(f_buf) > 3: f_buf.pop(0)

            f_tm1 = f_buf[0]
            f_t   = f_buf[1] if len(f_buf) >= 2 else f_buf[0]
            f_tp1 = f_buf[2] if len(f_buf) == 3 else f_t

            def norm_f(f):
                return cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2RGB), (640, 640)).astype(np.float32) / 255.0

            t_in = np.concatenate([norm_f(f_tm1), norm_f(f_t), norm_f(f_tp1)], axis=-1)[np.newaxis, ...]
            preds = model(t_in, training=False)
            scores, bboxes = decode_detections(preds["heatmap"], preds["offset"], preds["size"])
            conf = float(scores[0].numpy())
            box_n = bboxes[0].numpy()
            abs_box = [box_n[0]*orig_w, box_n[1]*orig_h, box_n[2]*orig_w, box_n[3]*orig_h]

            final_box, state, spd = tracker.update(abs_box, conf >= CONFIDENCE_THRESHOLD)

            disp_v = cv2.resize(f_t, (out_w, out_h))
            sx, sy = out_w / orig_w, out_h / orig_h
            if final_box is not None:
                scaled = [final_box[0]*sx, final_box[1]*sy, final_box[2]*sx, final_box[3]*sy]
                draw_hud_reticle(disp_v, scaled, conf, spd, state)

            diff = cv2.absdiff(f_t, f_tm1)
            diff_gray = cv2.normalize(cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY), None, 0, 255, cv2.NORM_MINMAX)
            disp_aux = cv2.resize(cv2.applyColorMap(diff_gray, cv2.COLORMAP_JET), (out_w, out_h))
            if final_box is not None:
                draw_hud_reticle(disp_aux, scaled, conf, spd, state)

            cv2.putText(disp_v, "CAM 1: VISIBLE RGB", (15, out_h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 1)
            cv2.putText(disp_aux, "CAM 2: MOTION HEATMAP", (15, out_h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 1)

            hud = np.zeros((60, canvas_w, 3), dtype=np.uint8)
            hud[:, :] = (20, 25, 30)
            cv2.putText(hud, "ANTI-UAV REALTIME TACTICAL TRACKING", (20, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)
            cv2.putText(hud, f"Frame {idx+1}/{tot_f} | Speed: {spd:.1f} px/f", (20, 48), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1)
            stat_col = (0, 255, 0) if state == "DETECTED" else ((0, 215, 255) if state == "TRACKED_EMA" else (100, 100, 100))
            cv2.putText(hud, f"STATUS: [{state}]", (canvas_w - 280, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.65, stat_col, 2)

            writer.write(np.vstack([hud, np.hstack([disp_v, disp_aux])]))

        cap.release()
        writer.release()

        # Convert to H.264
        os.system(f'ffmpeg -y -i "{raw_avi}" -vcodec libx264 -pix_fmt yuv420p -crf 23 "{out_demo_mp4}" -loglevel error')
        if os.path.exists(raw_avi): os.remove(raw_avi)

        print(f"✅ Render complete: {out_demo_mp4}")
        if os.path.exists(out_demo_mp4):
            mp4_bytes = open(out_demo_mp4, 'rb').read()
            data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()
            display(HTML(f'''
                <video width="800" controls style="border-radius: 8px; box-shadow: 0 4px 10px rgba(0,0,0,0.4);">
                    <source src="{data_url}" type="video/mp4">
                </video>
            '''))